In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ocha_stratus as stratus
import pandas as pd
from dotenv import load_dotenv
from src.constants import STATE_CONFIG
from src.datasources import glofas
from src.utils.rp_calc import calculate_one_group_rp

load_dotenv()

STATE = "Adamawa"
cfg = STATE_CONFIG[STATE]
FIGURES_DIR = "figures"

ANALYSIS_START_YEAR = 2003
ANALYSIS_END_YEAR = 2024
WET_MONTHS = [8, 9, 10, 11]
RP_LEVELS = [3, 5]
MAX_LEADTIME = 16
RP_COLORS = {3: "#74ADD1", 5: "#D73027"}

## Load data

In [8]:
# GloFAS reforecast ensemble
df_gf_rf = stratus.load_parquet_from_blob(cfg["glofas_reforecast_blob"])
df_gf_rf["time"] = pd.to_datetime(df_gf_rf["time"])
df_gf_rf["valid_time"] = pd.to_datetime(df_gf_rf["valid_time"])
df_gf_rf = df_gf_rf[df_gf_rf["leadtime"] <= MAX_LEADTIME]

# Floodscan pixels
df_fs = stratus.load_parquet_from_blob(cfg["floodscan_blob"])
df_fs["date"] = pd.to_datetime(df_fs["date"])

## Floodscan return period thresholds

Annual wet-season (Aug–Nov) maximum mean SFED across all riverine pixels, used to classify flood
years at each RP level. Return periods are computed from the full Floodscan record (1998–2025) for
robust threshold estimation. Performance evaluation is restricted to years where GloFAS reforecast
data exists (2003–2022).

The 4-year RP level is excluded because the Floodscan years that distinguish it from the 5-year
RP level (1999, 2023, 2024, 2025) all fall outside the GloFAS reforecast window, making the two
event sets identical within the evaluation period.

In [9]:
df_fs_daily = (
    df_fs.groupby("date")["SFED"].mean()
    .reset_index()
    .rename(columns={"SFED": "sfed_mean"})
)
df_fs_daily["date"] = pd.to_datetime(df_fs_daily["date"])
df_fs_daily = df_fs_daily[df_fs_daily["date"].dt.month.isin(WET_MONTHS)]

df_fs_annual = (
    df_fs_daily.assign(year=df_fs_daily["date"].dt.year)
    .groupby("year")["sfed_mean"].max()
    .reset_index()
)
df_fs_rp = calculate_one_group_rp(df_fs_annual, col_name="sfed_mean", ascending=False)

print(f"Floodscan record: {int(df_fs_annual['year'].min())}\u2013{int(df_fs_annual['year'].max())} ({len(df_fs_annual)} years)\n")
for rp in RP_LEVELS:
    thresh = df_fs_rp[df_fs_rp["sfed_mean_rp"] >= rp]["sfed_mean"].min()
    events = sorted(df_fs_rp[df_fs_rp["sfed_mean_rp"] >= rp]["year"].tolist())
    print(f"  {rp}-year RP: threshold \u2248 {thresh:.4f},  event years = {events}")

Floodscan record: 1998–2025 (28 years)

  3-year RP: threshold ≈ 0.1504,  event years = [1999, 2003, 2012, 2015, 2018, 2022, 2023, 2024, 2025]
  4-year RP: threshold ≈ 0.1690,  event years = [1999, 2012, 2015, 2022, 2023, 2024, 2025]
  5-year RP: threshold ≈ 0.2046,  event years = [2012, 2015, 2022, 2023, 2025]


## GloFAS reforecast preparation

Ensemble mean discharge per (valid_time, leadtime). Cumulative max is computed within each year
across leadtimes: at leadtime lt, `cum_max` is the highest ensemble-mean forecast seen at any
leadtime \u2264 lt during the wet season. A year is activated if `cum_max` exceeds the trigger
threshold (`glofas_thresh` from STATE_CONFIG).

In [10]:
df_gf_rf_mean = (
    df_gf_rf.groupby(["valid_time", "leadtime"])["dis24"]
    .mean()
    .reset_index()
)

rf_wet = df_gf_rf_mean[
    df_gf_rf_mean["valid_time"].dt.month.isin(WET_MONTHS)
    & df_gf_rf_mean["valid_time"].dt.year.between(ANALYSIS_START_YEAR, ANALYSIS_END_YEAR)
].copy()
rf_wet["year"] = rf_wet["valid_time"].dt.year

rf_annual = (
    rf_wet.groupby(["year", "leadtime"])["dis24"].max()
    .reset_index()
    .sort_values(["year", "leadtime"])
)
rf_annual["cum_max"] = rf_annual.groupby("year")["dis24"].cummax()

years_in_rf = sorted(rf_annual["year"].unique())
print(f"GloFAS reforecast: {years_in_rf[0]}\u2013{years_in_rf[-1]} ({len(years_in_rf)} years)")
print(f"Leadtime range   : {int(rf_annual['leadtime'].min())}\u2013{int(rf_annual['leadtime'].max())} days")
print(f"Trigger threshold: {cfg['glofas_thresh']:,} m\u00b3/s")

GloFAS reforecast: 2003–2022 (20 years)
Leadtime range   : 1–16 days
Trigger threshold: 3,132 m³/s


## Performance evaluation

For each (leadtime, RP level), years where GloFAS reforecast exists are classified into:

- **TP**: GloFAS activated AND Floodscan event year
- **FP**: GloFAS activated AND NOT Floodscan event year (false alarm)
- **FN**: GloFAS did NOT activate AND Floodscan event year (miss)
- **TN**: GloFAS did NOT activate AND NOT Floodscan event year

Metrics:

| Metric | Formula |
|---|---|
| Accuracy | (TP + TN) / N |
| Detection rate (POD) | TP / (TP + FN) |
| False alarm ratio (FAR) | FP / (TP + FP) |
| Precision | TP / (TP + FP) |
| F1 score | 2 \u00d7 Precision \u00d7 POD / (Precision + POD) |

Note: FAR and Precision are complementary (FAR + Precision = 1) \u2014 FAR answers \u201cof all activations,
how many were wrong?\u201d while Precision answers \u201chow many were right?\u201d

In [11]:
def trigger_performance(rf_annual, df_fs_rp, rp_levels, thresh):
    """Accuracy, detection rate, FAR, precision, and F1 vs leadtime for each RP level.

    Truth: Floodscan annual wet-season SFED >= N-year RP threshold.
    Trigger: GloFAS cumulative max reforecast at leadtime <= lt >= thresh (once per year).
    """
    all_fs_years = set(df_fs_rp["year"])
    fs_event_years = {
        rp: set(df_fs_rp[df_fs_rp["sfed_mean_rp"] >= rp]["year"])
        for rp in rp_levels
    }

    results = {}
    for rp in rp_levels:
        records = []
        for lt, grp in rf_annual.groupby("leadtime"):
            merged = grp.set_index("year")[["cum_max"]]
            merged = merged[merged.index.isin(all_fs_years)].copy()
            merged["event"] = merged.index.isin(fs_event_years[rp]).astype(int)
            merged["trigger"] = (merged["cum_max"] > thresh).astype(int)

            tp = int(((merged["trigger"] == 1) & (merged["event"] == 1)).sum())
            fp = int(((merged["trigger"] == 1) & (merged["event"] == 0)).sum())
            fn = int(((merged["trigger"] == 0) & (merged["event"] == 1)).sum())
            tn = int(((merged["trigger"] == 0) & (merged["event"] == 0)).sum())
            n = tp + fp + fn + tn

            accuracy = (tp + tn) / n if n > 0 else np.nan
            detection_rate = tp / (tp + fn) if (tp + fn) > 0 else np.nan
            far = fp / (tp + fp) if (tp + fp) > 0 else np.nan
            precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
            f1_denom = (precision or 0) + (detection_rate or 0)
            f1 = (2 * precision * detection_rate / f1_denom
                  if (precision is not None and detection_rate is not None and f1_denom > 0)
                  else np.nan)

            records.append({
                "leadtime": lt,
                "accuracy": accuracy,
                "detection_rate": detection_rate,
                "far": far,
                "precision": precision,
                "f1": f1,
                "tp": tp, "fp": fp, "fn": fn, "tn": tn, "n": n,
            })
        results[rp] = pd.DataFrame(records).set_index("leadtime")
    return results


perf = trigger_performance(rf_annual, df_fs_rp, RP_LEVELS, thresh=cfg["glofas_thresh"])

for rp in RP_LEVELS:
    n_events = len(set(df_fs_rp[df_fs_rp["sfed_mean_rp"] >= rp]["year"]) & set(years_in_rf))
    print(f"\n--- {rp}-year RP  ({n_events} event years within GloFAS period) ---")
    print(perf[rp][["accuracy", "detection_rate", "far", "precision", "f1",
                     "tp", "fp", "fn", "tn"]].round(2).to_string())


--- 3-year RP  (5 event years within GloFAS period) ---
          accuracy  detection_rate   far  precision    f1  tp  fp  fn  tn
leadtime                                                                 
1             0.85             0.4  0.00       1.00  0.57   2   0   3  15
2             0.85             0.4  0.00       1.00  0.57   2   0   3  15
3             0.85             0.4  0.00       1.00  0.57   2   0   3  15
4             0.85             0.4  0.00       1.00  0.57   2   0   3  15
5             0.85             0.4  0.00       1.00  0.57   2   0   3  15
6             0.80             0.4  0.33       0.67  0.50   2   1   3  14
7             0.75             0.4  0.50       0.50  0.44   2   2   3  13
8             0.80             0.6  0.40       0.60  0.60   3   2   2  13
9             0.80             0.6  0.40       0.60  0.60   3   2   2  13
10            0.80             0.6  0.40       0.60  0.60   3   2   2  13
11            0.75             0.6  0.50       0.50  0.

## Performance plots

Each line shows how a metric varies across leadtime for a given Floodscan RP event definition.
The shaded region indicates the current GloFAS action leadtime window (\u2264 5 days for Adamawa).

In [ ]:
action_lt = cfg.get("glofas_leadtime_action")

RP_LINESTYLES = {3: "-", 5: "--"}

metrics = [
    ("accuracy",       "Accuracy ((TP+TN) / N)"),
    ("detection_rate", "Detection rate / POD (TP / (TP+FN))"),
    ("far",            "False alarm ratio (FP / (TP+FP))"),
    ("precision",      "Precision (TP / (TP+FP))"),
    ("f1",             "F1 score"),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes_flat = axes.flatten()

for i, (metric, ylabel) in enumerate(metrics):
    ax = axes_flat[i]
    if action_lt is not None:
        ax.axvspan(0, action_lt, facecolor="#007CE0", edgecolor="none", alpha=0.08, zorder=0)
    for rp in RP_LEVELS:
        df = perf[rp]
        if not df.empty and metric in df.columns:
            ax.plot(
                df.index, df[metric],
                color=RP_COLORS[rp],
                linestyle=RP_LINESTYLES[rp],
                label=f"{rp}-yr RP",
            )
    ax.set_xlabel("Leadtime (days)")
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, 1.05)
    ax.set_title(f"{ylabel.split(' (')[0]} — GloFAS vs Floodscan events — {STATE}")
    ax.legend(fontsize=9)
    ax.grid(axis="y", linewidth=0.5, alpha=0.5)

axes_flat[5].set_visible(False)

plt.tight_layout()
plt.savefig(
    f"{FIGURES_DIR}/{STATE.lower()}_trigger_performance_vs_floodscan.png",
    dpi=200, bbox_inches="tight",
)
plt.show()